In [2]:
pip install tokenizers sentencepiece


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from tokenizers import Tokenizer
from tokenizers.models import BPE, WordPiece
from tokenizers.trainers import BpeTrainer, WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace
from collections import Counter
import sentencepiece as spm
import os


In [4]:
# --------------------------------------------------
# Step 1: Create Sample Corpus
# --------------------------------------------------

corpus = [
    "Machine learning enables computers to learn from data",
    "Deep learning is a subset of machine learning",
    "Large language models use tokenization techniques",
    "Tokenization converts text into smaller units called tokens",
    "Artificial intelligence is transforming industries",
    "Transformers are widely used in natural language processing"
]

with open("corpus.txt", "w", encoding="utf-8") as f:
    for line in corpus:
        f.write(line + "\n")

print("Corpus file exists:", os.path.exists("corpus.txt"))


Corpus file exists: True


In [5]:
# --------------------------------------------------
# Step 2: Train BPE Tokenizer
# --------------------------------------------------

bpe = Tokenizer(BPE(unk_token="[UNK]"))
bpe.pre_tokenizer = Whitespace()

bpe.train(
    ["corpus.txt"],
    BpeTrainer(
        vocab_size=50,
        special_tokens=["[UNK]"]
    )
)



In [5]:
# --------------------------------------------------
# Step 3: Train WordPiece Tokenizer
# --------------------------------------------------

wp = Tokenizer(WordPiece(unk_token="[UNK]"))
wp.pre_tokenizer = Whitespace()

wp.train(
    ["corpus.txt"],
    WordPieceTrainer(
        vocab_size=50,
        special_tokens=["[UNK]"]
    )
)


In [6]:
# --------------------------------------------------
# Step 4: Train SentencePiece Tokenizer
# --------------------------------------------------

spm.SentencePieceTrainer.train(
    input="corpus.txt",
    model_prefix="sp",
    vocab_size=50,
    model_type="unigram",
    hard_vocab_limit=False,
    bos_id=-1,
    eos_id=-1
)

sp = spm.SentencePieceProcessor()
sp.load("sp.model")


True

In [14]:
# --------------------------------------------------
# Step 5: Vocabulary Generation
# --------------------------------------------------

print("\n===== BPE Vocabulary =====")
bpe_vocab = list(bpe.get_vocab().keys())
print(bpe_vocab[:20])
print("Vocabulary Size:", len(bpe_vocab))

print("\n===== WordPiece Vocabulary =====")
wp_vocab = list(wp.get_vocab().keys())
print(wp_vocab[:20])
print("Vocabulary Size:", len(wp_vocab))

print("\n===== SentencePiece Vocabulary =====")
for i in range(min(20, sp.get_piece_size())):
    print(sp.id_to_piece(i))
print("Vocabulary Size:", sp.get_piece_size())


===== BPE Vocabulary =====
['learn', 'pu', 'enables', 'ter', 'Ma', 'ters', 'l', 'p', 'compu', 's', 'en', 'les', 'ar', 'm', 'at', 'Mach', 'data', 'to', 'com', 'er']
Vocabulary Size: 50

===== WordPiece Vocabulary =====
['##b', 'y', '##t', '##o', '##a', '##s', '##p', 'f', 'D', 'c', 'A', '##u', 'h', '##z', '##h', '##e', 'a', 'q', '##x', 'p']
Vocabulary Size: 55

===== SentencePiece Vocabulary =====
<unk>
▁
s
i
e
a
t
u
r
o
ge
d
at
n
m
ch
f
p
▁learning
on
Vocabulary Size: 50


In [8]:
# --------------------------------------------------
# Step 6: Tokenization Example
# --------------------------------------------------

text = "Large language models use tokenization"

print("\n===== Tokenization Example =====")

print("\nBPE Tokens")
print(bpe.encode(text).tokens)

print("\nWordPiece Tokens")
print(wp.encode(text).tokens)

print("\nSentencePiece Tokens")
print(sp.encode_as_pieces(text))



===== Tokenization Example =====

BPE Tokens
['L', 'ar', 'ge', 'l', 'an', 'g', 'u', 'a', 'ge', 'm', 'o', 'd', 'e', 'l', 's', 'u', 'se', 'to', 'ken', 'i', 'z', 'at', 'i', 'on']

WordPiece Tokens
['L', '##a', '##r', '##g', '##e', 'l', '##a', '##n', '##g', '##u', '##a', '##g', '##e', 'm', '##o', '##d', '##e', '##l', '##s', 'u', '##s', '##e', 't', '##o', '##k', '##e', '##n', '##i', '##z', '##a', '##t', '##i', '##o', '##n']

SentencePiece Tokens
['▁', 'L', 'a', 'r', 'ge', '▁l', 'a', 'ng', 'u', 'a', 'ge', '▁', 'm', 'o', 'del', 's', '▁use', '▁token', 'i', 'z', 'at', 'i', 'on']


In [9]:
# --------------------------------------------------
# Step 7: Unknown Token Handling
# --------------------------------------------------

unknown = "Transformerization"

print("\n===== Unknown Token Handling =====")

print("Input:", unknown)

print("\nBPE")
print(bpe.encode(unknown).tokens)

print("\nWordPiece")
print(wp.encode(unknown).tokens)

print("\nSentencePiece")
print(sp.encode_as_pieces(unknown))


===== Unknown Token Handling =====
Input: Transformerization

BPE
['T', 'r', 'an', 's', 'f', 'o', 'r', 'm', 'e', 'r', 'i', 'z', 'at', 'i', 'on']

WordPiece
['T', '##r', '##a', '##n', '##s', '##f', '##o', '##r', '##m', '##e', '##r', '##i', '##z', '##a', '##t', '##i', '##o', '##n']

SentencePiece
['▁', 'T', 'ransform', 'er', 'i', 'z', 'at', 'i', 'on']


In [10]:
# --------------------------------------------------
# Step 8: Token Distribution Analysis
# --------------------------------------------------

print("\n===== Token Distribution Analysis =====")

bpe_tokens = bpe.encode(" ".join(corpus)).tokens
wp_tokens = wp.encode(" ".join(corpus)).tokens
sp_tokens = sp.encode_as_pieces(" ".join(corpus))

print("\nTop 5 BPE Tokens")
print(Counter(bpe_tokens).most_common(5))

print("\nTop 5 WordPiece Tokens")
print(Counter(wp_tokens).most_common(5))

print("\nTop 5 SentencePiece Tokens")
print(Counter(sp_tokens).most_common(5))



===== Token Distribution Analysis =====

Top 5 BPE Tokens
[('s', 15), ('i', 14), ('r', 13), ('e', 10), ('u', 10)]

Top 5 WordPiece Tokens
[('##e', 34), ('##n', 29), ('##a', 23), ('##i', 19), ('##r', 19)]

Top 5 SentencePiece Tokens
[('▁', 28), ('s', 17), ('i', 16), ('e', 13), ('a', 12)]


In [11]:
# --------------------------------------------------
# Step 9: Context Window Utilization
# --------------------------------------------------

print("\n===== Context Window Utilization =====")

bpe_count = len(bpe.encode(text).tokens)
wp_count = len(wp.encode(text).tokens)
sp_count = len(sp.encode_as_pieces(text))

print("BPE Token Count:", bpe_count)
print("WordPiece Token Count:", wp_count)
print("SentencePiece Token Count:", sp_count)

minimum = min(bpe_count, wp_count, sp_count)

print("\nMost Efficient Tokenizer:")

if minimum == bpe_count:
    print("BPE")
elif minimum == wp_count:
    print("WordPiece")
else:
    print("SentencePiece")



===== Context Window Utilization =====
BPE Token Count: 24
WordPiece Token Count: 34
SentencePiece Token Count: 23

Most Efficient Tokenizer:
SentencePiece
